In [1]:
import os
import numpy as np
from astropy.io import fits

In [2]:
# ==================================================
# INPUTS
# ==================================================

science_folder = "/Users/sujoy7471/Desktop/TIFR/4th Sem/Astro2/Observations/AstrData/11May_Astro2/Data/2026-04-27_10_11_33Z"

bias_file = "/Users/sujoy7471/Desktop/TIFR/4th Sem/Astro2/Observations/AstrData/11May_Astro2/Bias/2026-04-27_09_04_38Z/Light_4_00mm_0.001sec_Bin1_16.5C_gain0_2026-04-27_090443_frame0003.fit"

dark_file = "/Users/sujoy7471/Desktop/TIFR/4th Sem/Astro2/Observations/AstrData/11May_Astro2/Dark/2026-04-27_12_52_13Z/Light_4_00mm_0.035sec_Bin1_-5.8C_gain0_2026-04-27_125220_frame0006.fit"

flat_file = "/Users/sujoy7471/Desktop/TIFR/4th Sem/Astro2/Observations/AstrData/11May_Astro2/Flat/2026-04-27_13_02_33Z/Light_4_00mm_0.035sec_Bin1_-4.1C_gain0_2026-04-27_130238_frame0003.fit"

output_file = "/Users/sujoy7471/Desktop/TIFR/4th Sem/Astro2/Observations/AstrData/11May_Astro2/reduced_data/May11_10_11_33_reduced.fits"

# ==================================================
# LOAD CALIBRATION FRAMES
# ==================================================

bias = fits.getdata(bias_file).astype(float)
dark = fits.getdata(dark_file).astype(float)
flat = fits.getdata(flat_file).astype(float)

flat = flat / np.median(flat)
flat[flat == 0] = 1

# ==================================================
# REDUCE ALL FRAMES
# ==================================================

reduced_frames = []

files = sorted([
    f for f in os.listdir(science_folder)
    if f.endswith((".fits", ".fit"))
])

print(f"Found {len(files)} files")

for i, file in enumerate(files):

    infile = os.path.join(science_folder, file)

    with fits.open(infile) as hdul:

        data = hdul[0].data.astype(float)

        reduced = (data - bias - dark) / flat

        reduced_frames.append(reduced)

        if i == 0:
            header = hdul[0].header

    print(f"Reduced: {file}")

# ==================================================
# STACK
# ==================================================

stack = np.array(reduced_frames)

# Average combine
combined = np.mean(stack, axis=0)

# OR use median combine (recommended)
# combined = np.median(stack, axis=0)

# ==================================================
# SAVE
# ==================================================

fits.writeto(
    output_file,
    combined,
    header=header,
    overwrite=True
)

print(f"\nSaved combined image to:\n{output_file}")

Found 5 files
Reduced: Light_4_00mm_0.035sec_Bin1_-2.9C_gain0_2026-04-27_101139_frame0005.fit
Reduced: Light_4_00mm_0.035sec_Bin1_-3.0C_gain0_2026-04-27_101136_frame0001.fit
Reduced: Light_4_00mm_0.035sec_Bin1_-3.0C_gain0_2026-04-27_101137_frame0002.fit
Reduced: Light_4_00mm_0.035sec_Bin1_-3.0C_gain0_2026-04-27_101137_frame0003.fit
Reduced: Light_4_00mm_0.035sec_Bin1_-3.0C_gain0_2026-04-27_101138_frame0004.fit

Saved combined image to:
/Users/sujoy7471/Desktop/TIFR/4th Sem/Astro2/Observations/AstrData/11May_Astro2/reduced_data/May11_10_11_33_reduced.fits
